# Documentation

The sections of this chapter explained every function that they used. This
courtesy cannot last. Every project soon needs a function that the book
never mentions, or uses a familiar one in an unfamiliar way, and no book can
keep up: the main module of a deep learning framework defines several
hundred to a thousand public names, and new releases add names and deprecate
others. What the book can teach instead is how to find out for ourselves,
reliably, what a function does.

Reliably is the operative word, because a call can look harmless and still
do something other than what we meant. Suppose that we need samples from a
gamma distribution, a distribution of positive numbers often used for
waiting times, with shape $k = 2$ and rate $\beta = 3$. Its mean is
$k / \beta = 2/3$ and its variance $k / \beta^2 = 2/9$. Libraries disagree
on how such a distribution is described: some take the rate $\beta$, others
the scale $1/\beta$, and some neither. A call that passes the numbers 2 and
3 runs either way, but its samples have mean $2/3$ under one reading and
$2 \cdot 3 = 6$ under the other. Nothing in the call tells us which reading
applies. The function's signature and documentation do.

Instead of a list of functions, this section gives a procedure of four
steps, shown in the figure. We discover the names that a
library offers, inspect the arguments and the documentation of a candidate,
read the reference pages or the source code when the documentation leaves a
question open, and verify the behavior with a small example whose result we
predicted. All four steps work inside a notebook, and by the end of the
section they have drawn our gamma samples, with the right mean.

![Four steps for finding out what a function does: discover the names that a module defines, inspect the signature and docstring of a candidate, read the reference documentation or the source code, and verify the behavior with a small example. A result that contradicts the prediction, or answers a question that the prediction left open, sends us back to read further.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/lookup-api-discovery-loop.svg)

Besides the framework, we import three helpers from Python's standard
library: `pprint` prints long lists compactly, `inspect` reads the
signature of a function, and `textwrap` wraps long lines.

In [ ]:
import inspect
import random
import textwrap
from pprint import pprint
import jax

## Discovering Names

Our first question is whether the library can sample from a gamma
distribution at all, and under which name. We know roughly where to look,
in the module for random numbers, but not what its functions are called.
The built-in function `dir` lists the names that a module defines. Names that begin with an underscore
belong to the internals of the module, so we drop them and print the first
twenty of the rest:

In [ ]:
pprint([name for name in dir(jax.random)
        if not name.startswith('_')][:20], compact=True)

The names are samplers, such as `bernoulli`, `beta`, `cauchy`,
`exponential`, and `gamma`, along with `Key`, the type of a random key, and
`PRNGKey`, an older function that creates the key every JAX sampler takes.
This book uses its successor, `jax.random.key`, which comes later in the
alphabet.

A notebook offers the same list while we type: writing the name of a module
followed by a dot and pressing Tab lists the names that the module defines,
narrowed to those that match the letters typed next. When we do not know
which module to look in, the search box of the framework's online reference
searches all of them.

## Reading Documentation

The list has an entry for the gamma distribution. To find out how to call
it, the built-in function `help` prints its *docstring*, the documentation
written into its code. A docstring says what the function does, what each
argument means, and what the function returns, and it often ends with
examples. The first lines show the function's *signature*, the list of
arguments that it accepts. Three conventions of Python are all we need to
read one. An argument written as `name=value` has a default value and may be
omitted. Arguments that follow a lone `*` can be passed only by keyword. An
argument written as `*name` collects any number of positional arguments;
our sampler has none of these, but the exercises meet one.

The whole docstring is worth reading once in a notebook, but it is long, and
our question is narrow: does the sampler take a rate or a scale? We
therefore print the signature, which `inspect.signature` returns, and only
those lines of the docstring that mention a rate or a scale. Since we print
single lines, a sentence may arrive cut off; the whole of it is one `help`
call away.

In [ ]:
def doc_lines(f, *words):
    """Print the signature of f and the lines of its docstring with a word."""
    print(textwrap.fill(f'{f.__name__}{inspect.signature(f)}', 79,
                        subsequent_indent='    '))
    for line in inspect.getdoc(f).splitlines():
        if any(word in line for word in words):
            print('   ', line.strip())

In [ ]:
doc_lines(jax.random.gamma, 'rate', 'scale')

With the signature and these lines, we can predict what a call does before
we run it:

The signature `gamma(key, a, shape=None, ...)`, printed with its type
hints, has neither a rate nor a scale. The sampler draws from the gamma
distribution with shape parameter `a` and rate 1, and the docstring tells
us to divide the samples by the rate. We predict that
`jax.random.gamma(key, 2.0, (10000,)) / 3` has mean $2/3$ and variance
$2/9$.

In a notebook, two shortcuts show the same documentation. Appending `?` to
a name shows the docstring in a separate pane, and appending `??` also
shows the source code of the function. The source answers
questions that a docstring leaves open, such as how an argument is converted
or checked. Unfortunately, only functions written in Python have source code
to show. Many core functions are compiled from C++, and for those `??` shows
the docstring alone.

When the docstring is not enough, the framework's reference documentation
describes every public function on a page of its own, and its tutorials show
the functions in use.

For JAX, these are the
[API reference](https://docs.jax.dev/en/latest/jax.html) and the
[tutorials](https://docs.jax.dev/en/latest/tutorials.html).

## Verifying Behavior

Our reading of a docstring can be wrong, and occasionally the docstring
itself is out of date. A small example settles the question, provided that
we predict its result before we run it, so that the run could prove us
wrong. A run that merely completes without an error proves little. Our
prediction can fail: had we read the rate as a scale, the samples would have
mean 6 instead of $2/3$. We draw 10,000 samples and compare their mean and
variance with $2/3 \approx 0.667$ and $2/9 \approx 0.222$. The mean of
10,000 independent draws has standard error $\sqrt{(2/9) / 10{,}000}
\approx 0.005$ (that section), so the sample mean should lie within
about $0.01$ of $2/3$:

In [ ]:
key = jax.random.key(random.getrandbits(32))  # A new seed on each run
samples = jax.random.gamma(key, 2.0, (10000,)) / 3  # k = 2, then rate 3
samples.mean(), samples.var()

The prediction holds: the mean lies close to $2/3$ and the variance close to
$2/9$, far from the values that the other reading would give. When a result
contradicts the prediction, or answers a question that the prediction left
open, we go around the loop again: we read further, in the reference
documentation or the source, and test the new prediction.

Coding assistants answer many such questions directly. Asked for gamma
samples with a given rate, an assistant proposes a call, and the call
usually looks plausible. Such a suggestion is a candidate, and it goes
through the same steps as a name we found ourselves: we check that the name
exists, for example with `dir` or `hasattr`, read its signature, predict the
result, and run a small example. The last step matters most, because a
wrong suggestion need not fail: it can name a function that exists, fit
its signature, and run, yet compute something else.

With this procedure the tools of the chapter are complete: tensors, tables,
linear algebra, derivatives, probability, and a way to find what the book
leaves out. that section uses them to build and train a first
model.

## Summary

No text can record the details of every function in a deep learning framework,
and a call that looks harmless may read its arguments differently than we
assume: the same two numbers describe a gamma distribution by its rate in
one library and by its scale in another. Four steps settle such questions from within a notebook. `dir` and
Tab completion discover the names that a module defines, and `help` and `?`
show the docstring of a candidate and its signature, which says what each
argument means. The reference documentation and `??` supply
the details that the docstring omits, and a small example whose result we
predicted verifies our reading. Suggestions from coding assistants are
candidates that go through the same steps.

## Exercises

1. [code] **Two integers or one tuple.** `jnp.ones` creates an array of
   ones. Read its signature with `help(jnp.ones)`, which begins
   `ones(shape, dtype=None, *, ...)`, and predict, before running anything,
   what each of `jnp.ones((2, 3))`, `jnp.ones(2, 3)`,
   `jnp.ones(shape=(2, 3))`, and `jnp.ones(size=(2, 3))` returns or raises.
   Then run the four calls and compare. For each call that fails, state
   which argument received the offending value and whether the error
   message says so.
1. [code] **Standard-deviation conventions.** For $n$ numbers whose
   squared deviations from their mean sum to $S$, a library computes the
   standard deviation either as $\sqrt{S/n}$ or as $\sqrt{S/(n-1)}$.
   that section standardized columns with the pandas method `std`.
   Compute the standard deviation of the values 1.0, 2.0, 3.0 and 4.0
   once with `pd.Series.std` and once with `jnp.std`. Use `help` on both
   functions to find out how each one sets the divisor, state the default
   divisor of each, and check both results against a hand computation.
   Then make the two results agree.
1. [code] **Checking a coding assistant's answer.** Ask a coding
   assistant the two questions below. Check each function it names in
   three steps: `dir` or `hasattr` shows that the name exists, `help` or
   `?` shows that the signature fits the suggested call, and a tiny
   example shows the claimed behavior.
    1. Ask how to split a JAX array into three equal parts along an axis.
    1. Ask for a single JAX function that reorders the axes of an array
       so that their lengths increase. Whatever the assistant answers,
       check each named function in the three steps, state which step, if
       any, it fails, and say whether the claimed behavior holds. If no
       suggestion passes, build the operation from functions that do, and
       test it on an array of shape $(4, 2, 3)$.

4. [code] **Reading the source.** The plotting function `d2l.plot`,
   defined in that section, accepts its data in several forms:
   a single tensor, a tensor of $x$-values with one tensor of $y$-values,
   or a tensor of $x$-values with several curves. Import `d2l` as at the
   start of this section, read the source of `d2l.plot` with `d2l.plot??` or your
   editor's go-to-definition, and find the lines that decide how many
   curves it draws and which values serve as their $x$-coordinates. Let
   `x` hold 50 evenly spaced values from $-2$ to $2$ (use `linspace`), and
   let `Y` be a matrix of shape $(3, 50)$.
   Predict from the source what `d2l.plot(x)`, `d2l.plot(x, x ** 2)`, and
   `d2l.plot(x, Y)` draw, in particular whether `Y` gives three curves of
   50 points or fifty curves of 3 points. Then run the three calls and
   compare.

1. [code] **Tests that can fail.** ● A verifying run is informative only if
   it would have given a different result had the claim been false.
    1. Find the framework's elementwise rounding function with `dir`, and
       design an input of at most four numbers on which three rules give
       three different results: round halves up, round halves away from
       zero, and round halves to the nearest even integer. Run the function
       on this input and state which rule the framework follows.
    1. A sort is *stable* if elements that compare equal keep their
       original order. Find the framework's `argsort`, and read what its
       docstring promises about stability. Then test stability on the input
       $(1, 0, 1, 0)$ and on 1,000 random integers from $\{0, 1\}$. Explain
       why a passing test on a short input cannot establish stability while
       a single failure refutes it, and state whether the behavior you
       observe agrees with the docstring.

[Discussions](https://d2l.discourse.group/t/17972)